# Nutrition5k: experimentos de los finalistas

Etapa 3 de los experimentos finales (rama `final-experiments`, imágenes cenitales). Los finalistas y su nivel de
fine-tuning se eligieron el 2026-11-01 con la semilla 0 de la etapa 2 (ver
`notes/decisions_and_concepts/final-experiments.md`):

| Finalista | Familia | Nivel |
|---|---|---|
| `v2` | mobilenet | `all` |
| `v4m` | mobilenet | `all` |
| `v4hm` | hybrid_mobile | `all` |
| `deit_t` | plain_vit | `all` (y 10 bloques, caso límite) |
| `dinov2_s` | plain_vit | 6 bloques |
| `swin_t` | hierarchical | `all` |

- Notebook fino: solo llama a `python -m nutrition5k train`. Cada run se guarda con los de su familia en
  `outputs/<familia>/<full|pilot>/<backbone>/<run>/`, así que los notebooks de familia también los ven.
- Receta común de la etapa 2: cabeza `512 · single 256`, D4, lr inicial 1e-4 (backbone) y 1e-3 (cabeza) con coseno
  hasta 0 en 80 épocas, AdamW con weight decay 0.01, BatchNorm congelada, dropout y drop path a 0, 224 px, CLS ⊕ media
  en los ViT.
- Volver a ejecutar una celda es seguro: los runs terminados se saltan y los interrumpidos se reanudan.
- Paso 1: semillas 1 y 2 de la receta. Paso 2: cambios de receta, uno a uno, con la semilla 0.
- Los resultados se analizan en `notebooks/results_general.ipynb`, con validación; test solo se reporta.

## Parámetros

In [ ]:
import os

FULL = True          # False: piloto 32/8 para pruebas rápidas
EPOCHS = 80
BATCH = 32
WORKERS = min(15, os.cpu_count()-2)  # procesos que decodifican las imágenes

FULL_FLAG = "--full" if FULL else ""
SPLIT = "full" if FULL else "pilot"

## Preparación del entorno

In [ ]:
import os
import sys
from pathlib import Path

# The volume is mounted at /workspace only on RunPod.
RUNPOD = Path("/workspace").is_dir()
if RUNPOD:
    # SSH sessions (VS Code Remote) do not inherit the pod's variables, such as the W&B secret.
    for variable in Path("/proc/1/environ").read_text().split("\0"):
        name, _, value = variable.partition("=")
        if name == "WANDB_API_KEY":
            os.environ.setdefault(name, value)
    # The template points the Hugging Face cache at the volume, where its symlinks break
    # (object storage); keep the pretrained weights on the pod's local disk instead.
    os.environ["HF_HOME"] = "/root/.cache/huggingface"

REPOSITORY_URL = "https://github.com/victorabellotorres/thesis-multimodal-health-ai.git"
# Local disk of the pod: fast, but lost when the pod ends.
REPOSITORY_DIR = Path("/root/thesis-multimodal-health-ai")
BRANCH = "final-experiments"
# Persistent storage: the volume mounted at /workspace.
PERSIST_ROOT = Path("/workspace/nutrition5k")

if RUNPOD:
    if (REPOSITORY_DIR / ".git").is_dir():
        !git -C {REPOSITORY_DIR} fetch
        !git -C {REPOSITORY_DIR} checkout {BRANCH}
        !git -C {REPOSITORY_DIR} pull --ff-only
    else:
        !git clone -b {BRANCH} {REPOSITORY_URL} {REPOSITORY_DIR}
    %cd {REPOSITORY_DIR}
    # The image's system Python is "externally managed" (PEP 668); the pod is disposable.
    %pip install -q --break-system-packages -e .[train]
elif Path.cwd().parent.name == "notebooks":
    # Local: the package is already installed in code/.venv; run from the repository root.
    os.chdir("../..")

# The kernel's interpreter, so `!` commands use the same environment locally (VS Code may not activate .venv).
PYTHON = sys.executable

if RUNPOD:
    # Results live on the volume so nothing is lost when the pod ends.
    (PERSIST_ROOT / "outputs").mkdir(parents=True, exist_ok=True)
    if not Path("outputs").exists():
        Path("outputs").symlink_to(PERSIST_ROOT / "outputs")
print("W&B:", "on" if os.environ.get("WANDB_API_KEY") else "off (only local files)")

In [ ]:
DATA_TAR = PERSIST_ROOT / f"data-{SPLIT}-overhead.tar"

if RUNPOD and DATA_TAR.is_file():
    !tar -xf {DATA_TAR}
else:
    !{PYTHON} -m nutrition5k fetch
    !{PYTHON} -m nutrition5k prepare {FULL_FLAG}
    if RUNPOD and _exit_code == 0:  # a failed prepare must not be cached as a partial .tar
        !tar -cf {DATA_TAR} data
!{PYTHON} -m nutrition5k preprocess  # excluded_dishes.csv and dish_groups.csv
!{PYTHON} -m nutrition5k check {FULL_FLAG}

In [ ]:
import subprocess

# The common recipe; train(..., **changes) replaces some of its values.
RECIPE = dict(heads="single", hidden=(512,), image_size=224, backbone_lr=1e-4, head_width=256, augment="d4",
              schedule="cosine", weight_decay=0.01, drop_path=0.0)


def train(backbone, unfreeze="all", pooling=None, seed=0, **changes):
    """One run of the common recipe with ``changes``; finished runs are skipped, interrupted ones resume.

    Stops the notebook if CUDA is gone (a host fault once made `train` fall back to CPU, 2026-10-28) or if
    the run fails.
    """
    if RUNPOD and subprocess.run([PYTHON, "-c", "import sys, torch; sys.exit(not torch.cuda.is_available())"]).returncode:
        raise RuntimeError("CUDA is not available on this pod")
    options = RECIPE | changes
    flags = " ".join(f"--hidden {size}" for size in options.pop("hidden"))
    flags += "".join(f" --{name.replace('_', '-')} {value}" for name, value in options.items())
    command = (
        f"{PYTHON} -m nutrition5k train --backbone {backbone} --unfreeze {unfreeze} {flags} "
        f"--epochs {EPOCHS} --batch-size {BATCH} --num-workers {WORKERS} --seed {seed} {FULL_FLAG}"
    )
    if pooling:
        command += f" --pooling {pooling}"
    print(command)
    get_ipython().system(command)
    if get_ipython().user_ns["_exit_code"]:
        raise RuntimeError(f"train failed: {command}")


## Paso 1: semillas 1 y 2 de la receta común

La semilla 0 ya está hecha (etapa 2). Con las tres se mide el ruido entre semillas y se recalcula el umbral de empate
antes de probar cambios de receta. `deit_t` con 10 bloques quedó 0.80 mejor que `all` con la semilla 0 (justo bajo el
umbral), así que también se repite. 14 runs, ~1.6 h en una RTX PRO 4500.

In [ ]:
FINALISTS = [("v2", "all", None), ("v4m", "all", None), ("v4hm", "all", None), ("deit_t", "all", "cls+mean"),
             ("deit_t", 10, "cls+mean"), ("dinov2_s", 6, "cls+mean"), ("swin_t", "all", None)]

for seed in (1, 2):
    for backbone, unfreeze, pooling in FINALISTS:
        train(backbone, unfreeze, pooling, seed)

## Paso 2: cambios de receta (semilla 0)

Cada run cambia una sola cosa de la receta en uno de los 6 finalistas (valores elegidos el 2026-11-01):

- lr del backbone 3e-5 y 3e-4 (factor ~3 a cada lado de 1e-4; la cabeza sigue en 1e-3).
- Cabeza: lineal sin trunk y `1024x1024 · single 256` (la más simple y la más grande del grid de la etapa 1).
- 256 px en `v4m` y `v4hm`, con pesos preentrenados a 256 (`v4m-r256`: misma receta e500; `v4hm-r256`: receta
  `ix_e550` de timm, la única de ImageNet-1k a 256).
- Regularización: weight decay 0.05 (en vez de 0.01) y drop path 0.1 (en vez de 0), por separado. El drop path
  sube linealmente hasta 0.1 en el último bloque, también en los bloques congelados de `dinov2_s`.

38 runs, en el orden de la lista (si el pod se corta, quedan cambios completos). Un cambio que en algún
finalista mejore la media de 3 semillas de la receta por más del umbral de criba (~1.0) se repite después con las
semillas 1 y 2.


In [ ]:
FINALISTS = [("v2", "all", None), ("v4m", "all", None), ("v4hm", "all", None), ("deit_t", "all", "cls+mean"),
             ("dinov2_s", 6, "cls+mean"), ("swin_t", "all", None)]
CHANGES = [
    dict(backbone_lr=3e-5),
    dict(backbone_lr=3e-4),
    dict(hidden=(0,), head_width=0),  # linear head, without trunk
    dict(hidden=(1024, 1024)),
    dict(weight_decay=0.05),
    dict(drop_path=0.1),
]

for changes in CHANGES[:4]:
    for backbone, unfreeze, pooling in FINALISTS:
        train(backbone, unfreeze, pooling, **changes)
for backbone in ("v4m-r256", "v4hm-r256"):
    train(backbone, image_size=256)
for changes in CHANGES[4:]:
    for backbone, unfreeze, pooling in FINALISTS:
        train(backbone, unfreeze, pooling, **changes)
